# Unsupervised Learning 

This notebook contains demo code for:
+ KMeans Clustering
+ DBSCAN
+ Apriori association rules


### Import Packages

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import kagglehub
from pathlib import Path
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

### KMeans Clustering

In [ ]:
path = kagglehub.dataset_download("vjchoudhary7/customer-segmentation-tutorial-in-python")
print("Path to dataset files:", path)

data_dir = Path(path)
print(list(data_dir.iterdir()))

file_path = data_dir / "Mall_Customers.csv"

#### Reading and Understanding the Dataset

In [ ]:
df1 = pd.read_csv(file_path)
print(df1.head())

In [ ]:
df1.info()

In [ ]:
print(df1.describe())

Notes:
+ 4 variables are available: `Gender`, `Age`, `Annual Income (k$)`, and `Spending Score (1-100)`.
+ The binary `Gender` variable is excluded from K-Means because scaling it does not remove its two-class structure; it could still dominate Euclidean distances and split clusters mainly by gender.

Variables:
+ Spending Score: score assigned by the mall based on customer behaviour and spending nature
+ Other variables are self-explanatory

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    x=df1['Annual Income (k$)'],
    y=df1['Spending Score (1-100)'],
    zorder=3
)
ax.set(
    title='Annual Income vs Spending Score',
    xlabel='Annual Income (k$)',
    ylabel='Spending Score (1-100)'
)
ax.grid(alpha=0.5, linestyle='--', zorder=1)
plt.tight_layout()
plt.show()

#### KMeans Demonstration

##### Preparing the Data

We want to prepare the data for KMeans clustering. 

There is not much to do, mainly we should scale the data, convert any type(str) variables into numerical values (only if it makes sense eg. binary, ordinal) and set the KMeans model.

Note: 
+ It is always better to scale your data if your model relies on Euclidean distance (eg. KNN, kmeans). 
+ A variable with a large scale can overwhelm other variables on a smaller scale. 

In [ ]:
# Use the three continuous variables for K-Means.
# Gender is binary; scaling it still leaves two distinct classes that can dominate distances.
df1_demo = df1[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].copy()

In [ ]:
print(df1_demo.head())

In [ ]:
# Scale the continuous features because K-Means relies on Euclidean distance.
scaler = StandardScaler()
df1_scaled = scaler.fit_transform(df1_demo)

K-Means parameters (for sklearn):
+ `n_clusters`: the number of clusters whose centroids you want to estimate
+ `random_state`: keeps the random choices reproducible
+ `n_init`: the number of centroid initialisations tried. With `random_state=42`, the same seed controls the reproducible sequence of initialisations; the run with the lowest inertia is kept. With `n_init="auto"`, scikit-learn uses one run when the initialisation is `k-means++`.
+ `tol`: convergence tolerance based on the movement of the cluster centres between iterations. A small movement means the algorithm has locally settled down; it does not prove that the globally optimal clustering has been found.


In [ ]:
# Create a K-Means instance, then fit the model to the scaled features.
kmeans = KMeans(n_clusters=5, random_state=42, n_init="auto")
kmeans.fit(df1_scaled)

In [ ]:
print("Cluster Labels:", kmeans.labels_) 

In [ ]:
df1_demo['Label'] = kmeans.labels_
print(df1_demo.head())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    x=df1_demo['Annual Income (k$)'],
    y=df1_demo['Spending Score (1-100)'],
    c=df1_demo['Label'],
    cmap='viridis',
    zorder=3
)

ax.set(
    title='K-Means Segments by Annual Income Against Spending Score',
    xlabel='Annual Income (k$)',
    ylabel='Spending Score (1-100)'
)
ax.grid(alpha=0.5, linestyle='--', zorder=1)
plt.tight_layout()
plt.show()

Why can the 2-D plot look unclear?

+ K-Means was fitted using three features, but this chart shows only two of them.
+ The clusters are groups of points in a 3-D feature space, so a 2-D projection can overlap points that are separated along the omitted spending-score dimension.
+ Plotting different feature pairs can reveal different parts of the same clustering; it does not change the fitted model.

##### 3D Plot

In [ ]:
plot_df = df1_demo.assign(Cluster=df1_demo['Label'].astype(str))

fig_3d = px.scatter_3d(
    plot_df,
    x='Age',
    y='Annual Income (k$)',
    z='Spending Score (1-100)',
    color='Cluster',
    color_discrete_sequence=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'],
    title='3D View of K-Means Customer Segments',
    labels={
        'Age': 'Age',
        'Annual Income (k$)': 'Annual Income (k$)',
        'Spending Score (1-100)': 'Spending Score (1-100)',
        'Cluster': 'Cluster'
    }
)

fig_3d.update_traces(marker={'size': 3, 'opacity': 0.6})

fig_3d.update_layout(
    width=900,
    height=700,
    legend_title_text='Cluster'
)
fig_3d.show()


In [ ]:
print("Centroids:\n", kmeans.cluster_centers_)

print('\n')

print("Centroids in Original Units:\n",scaler.inverse_transform(kmeans.cluster_centers_))


In [ ]:
new_customer = pd.DataFrame([{
    'Age': 23,
    'Annual Income (k$)': 40,
    'Spending Score (1-100)': 60
}])

new_scaled = scaler.transform(new_customer)
predicted_labels = kmeans.predict(new_scaled)
cluster_label = int(predicted_labels[0])

print(f"New Customer Belongs to Cluster {cluster_label}")

new_customer.assign(Cluster=cluster_label)

#### Plotting the Scree Plot

In [ ]:
cluster_range = range(1, 11)
inertias = []

for n in cluster_range:
    model = KMeans(n_clusters=n, random_state=42, n_init='auto')
    model.fit(df1_scaled)
    inertias.append(model.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cluster_range, inertias, marker='o')
ax.set(
    title='Scree Plot: Choosing the Number of Clusters',
    xlabel='Number of clusters (k)',
    ylabel='Inertia'
)
ax.set_xticks(list(cluster_range))
ax.grid(alpha=0.5, linestyle='--')

# The elbow is a visual judgement; this annotation marks the selected k=5 coordinate.
elbow_k = 5
elbow_inertia = inertias[list(cluster_range).index(elbow_k)]
ax.annotate(
    'Candidate elbow (k=5)',
    xy=(elbow_k, elbow_inertia),
    xytext=(elbow_k + 1, elbow_inertia + 40),
    arrowprops={'arrowstyle': '->', 'color': 'red'},
    color='red'
)

plt.tight_layout()
plt.show()

#### Using Silhouette Score to Choose k

The silhouette score measures how well each observation fits within its assigned cluster compared with the nearest neighbouring cluster. Scores range from -1 to 1, and higher values indicate better-separated, more cohesive clusters.

Because silhouette score cannot be calculated for a single cluster, we test values of `k` from 2 to 10 and identify the value with the highest average score. The scree plot and silhouette score can recommend different values because inertia always decreases as `k` increases, while silhouette rewards separation and cohesion. 

In [ ]:
from sklearn.metrics import silhouette_score

cluster_range = range(2, 11)
silhouette_scores = []

for n in cluster_range:
    model = KMeans(n_clusters=n, random_state=42, n_init='auto')
    labels = model.fit_predict(df1_scaled)
    silhouette_scores.append(silhouette_score(df1_scaled, labels))

best_k = list(cluster_range)[np.argmax(silhouette_scores)]
best_score = max(silhouette_scores)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cluster_range, silhouette_scores, marker='o', color='darkorange')
ax.axvline(best_k, color='red', linestyle='--', label=f'Best k = {best_k}')
ax.set(
    title='Using Silhouette Score to Choose K',
    xlabel='K',
    ylabel='Average silhouette score'
)
ax.set_xticks(list(cluster_range))
ax.grid(alpha=0.5, linestyle='--')
ax.legend()

plt.tight_layout()
plt.show()

print(f'Best k : {best_k}')
print(f'Highest avg score: {best_score:.3f}')

##### Better Visualisation

In [ ]:
## Obtain the columns that we want
df1_demo = df1[['Annual Income (k$)', 'Spending Score (1-100)']].copy()

## Scaling
df1_scaled = scaler.fit_transform(df1_demo)

## Fit the KMeans Model
kmeans = KMeans(n_clusters=5, random_state=42, n_init="auto")
kmeans.fit(df1_scaled)

## Get labels
df1_demo['Label'] = kmeans.labels_
print(df1_demo.head())

## Plot the Graph
fig, ax = plt.subplots(figsize=(8, 5))

scatter = ax.scatter(
    x=df1_demo['Annual Income (k$)'],
    y=df1_demo['Spending Score (1-100)'],
    c=df1_demo['Label'],
    cmap='viridis',
    zorder=3
)

ax.set(
    title='Customer Segments from KMeans',
    xlabel='Annual Income (k$)',
    ylabel='Spending Score (1-100)'
)
ax.grid(alpha=0.5, linestyle='--', zorder=1)
plt.tight_layout()
plt.show()

print("Centroids:\n", kmeans.cluster_centers_)

### DBSCAN

In [ ]:
path = kagglehub.dataset_download("berkayalan/sklearn-moons-data-set")
print("Path to dataset files:", path)

data_dir = Path(path)
print(list(data_dir.iterdir()))


file_path = data_dir / "cluster_moons.csv"

#### Reading and Understanding the dataset

In [ ]:
df2 = pd.read_csv(file_path)
print(df2.head())

Notes: This dataset just contains coordinates to half-circles in R2

#### What happens if we apply K-Means to this?

The two moon coordinates are already on comparable scales. Therefore, scaling is unnecessary here 


In [ ]:
df2_kmeans = df2.copy()

# The moon coordinates are already on comparable scales. No scaling is needed here,
# and leaving them unchanged preserves the geometry we want to compare with DBSCAN.
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
kmeans.fit(df2_kmeans[['X1', 'X2']])

df2_kmeans['Label'] = kmeans.labels_
print(df2_kmeans.head())

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    x=df2_kmeans['X1'],
    y=df2_kmeans['X2'],
    c=df2_kmeans['Label'],
    cmap='viridis',
    zorder=3
)
ax.set(title='Moons Clustered with K-Means', xlabel='X1', ylabel='X2')
ax.grid(alpha=0.5, linestyle='--', zorder=1)
plt.tight_layout()
plt.show()

print('Centroids:')
print(kmeans.cluster_centers_)

#### DBSCAN Demonstration

In [ ]:
df2_db = df2.copy()

DBSCAN parameters (for sklearn):
+ `eps`: the maximum distance between two samples for one to be considered in the neighbourhood of the other.
+ `min_samples`: the number of samples in a neighbourhood, including the point itself, for that point to be considered a core point.
+ `metric`: the metric used to calculate distances between samples; the default is Euclidean distance.

Other parameters include `algorithm`, `leaf_size`, `p`, and `n_jobs`, but the first three are the most relevant for this demonstration.


In [ ]:
# Create an instance of the DBSCAN model, then fit it to the moon coordinates.
db = DBSCAN(eps=0.2, min_samples=3, metric='euclidean')
db.fit(df2_db[['X1', 'X2']])

In [ ]:
labels = db.labels_
df2_db['Label'] = labels

# DBSCAN uses -1 for points classified as noise.
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int(np.sum(labels == -1))
print('The number of clusters (excluding noise):', n_clusters)
print('The number of noise points (label -1):', n_noise)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    x=df2_db['X1'],
    y=df2_db['X2'],
    c=df2_db['Label'],
    cmap='viridis',
    zorder=3
)
ax.set(title='Moons Clustered with DBSCAN', xlabel='X1', ylabel='X2')
ax.grid(alpha=0.5, linestyle='--', zorder=1)
plt.tight_layout()
plt.show()

##### Finding Optimal Epsilon With a k-distance Plot

In [ ]:
# Use min_samples + 1 neighbours so the selected distance is to the third other neighbour;
# sklearn's neighbourhood count includes the point itself.
min_samples = 3
number_of_neighbours = min_samples + 1
nearest_neighbors = NearestNeighbors(n_neighbors=number_of_neighbours)
nearest_neighbors.fit(df2)
distances, indices = nearest_neighbors.kneighbors(df2)

k_distances = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(k_distances)), k_distances)
ax.set(
    title='Third-Other-Neighbour Distance Plot for DBSCAN eps',
    xlabel='Data points sorted by distance',
    ylabel='Distance to third other neighbour'
)
ax.grid(alpha=0.5, linestyle='--')
ax.axhline(y=0.05, linestyle='--', color='red')
ax.annotate('Candidate eps', xy=(1200, 0.05), xytext=(0, 5), textcoords='offset points', color='red')
plt.tight_layout()
plt.show()


## Appendix

### Apriori

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder 

In [ ]:
path = kagglehub.dataset_download("heeraldedhia/groceries-dataset")
print("Path to dataset files:", path)

data_dir = Path(path)
print(list(data_dir.iterdir()))

file_path = data_dir / "Groceries_dataset.csv"


In [ ]:
df3 = pd.read_csv(file_path)
print(df3.head())

In [ ]:
df3.info()


#### Apriori Demonstration

Apriori identifies items that frequently occur together in the same grocery transaction. Here, each member's shopping trip on a particular date is treated as one transaction.

##### Preparing the Transactions

Apriori expects one row per transaction and one Boolean column per item. The encoded table records whether each item appeared during a shopping trip.

In [ ]:
transactions = (
    df3.groupby(['Member_number', 'Date'])['itemDescription']
    .apply(list)
    .tolist()
)

te = TransactionEncoder()
transaction_array = te.fit(transactions).transform(transactions)
df3_encoded = pd.DataFrame(transaction_array, columns=te.columns_)

print(f'Number of transactions: {len(df3_encoded)}')
print(f'Number of unique items: {df3_encoded.shape[1]}')


In [ ]:
## This step generates candidate itemsets and prunes the itemsets based on support threshold
frequent_itemsets = apriori(df3_encoded, min_support=0.005, use_colnames=True)
## This step creates the association rules
## Retains rules with confidence >= 0.1
rules = association_rules(frequent_itemsets, metric='confidence',min_threshold=0.1)

frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False).reset_index(drop=True)
rules = rules.sort_values(by='lift', ascending=False).reset_index(drop=True)

print('Most frequent itemsets:')
print(frequent_itemsets.head(10))
print('\n')
print('Strongest association rules by lift:')
## Note: Antecedents -> Consequents 
## Antecedent: The Given 
## Consequent: Outcome we are examining 
## In the slides we do Confidence(A->B), where A is the antecedent and B is the consequent
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

The `support` value measures how often an itemset appears in all transactions. `confidence` measures how often the consequent appears when the antecedent appears. `lift` compares the observed co-occurrence with what would be expected if the antecedent and consequent were independent:

+ `lift > 1` suggests a positive association.
+ `lift = 1` is consistent with independence.
+ `lift < 1` suggests a negative association.

Only one of the displayed rules has lift above 1, and even that lift is only slightly above 1. This dataset therefore does not show strong, practically meaningful associations in the selected rules. Many rules end in `whole milk` because it is the most frequent item. That makes confidence less informative for this consequent, so lift is the better measure for asking whether the pair occurs more often than its individual frequencies would predict.


In [ ]:
top_rules = rules.head(10).copy()
top_rules['rule'] = (
    top_rules['antecedents'].apply(lambda items: ', '.join(sorted(items)))
    + ' -> '
    + top_rules['consequents'].apply(lambda items: ', '.join(sorted(items)))
)

top_rules = top_rules.sort_values(by='lift')

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top_rules['rule'], top_rules['lift'], color='steelblue')
ax.set(title='Top Grocery Association Rules by Lift', xlabel='Lift', ylabel='Rule')
ax.grid(axis='x', alpha=0.5, linestyle='--')

plt.tight_layout()
plt.show()

The chart ranks the strongest rules by lift. Rules with larger lift values occur together more often than independence would suggest, but the values here are only slightly above 1, so the apparent associations should be interpreted cautiously.


#### Apriori Summary

- The dataset contains 14,963 member-date transactions and 167 unique grocery items.
- `whole milk`, `other vegetables`, and `rolls/buns` are the three most frequent individual items.
- Only one displayed rule has lift greater than 1, and its lift is approximately 1.12, so the dataset shows weak rather than strong evidence of meaningful product associations.
- `whole milk` appears as a consequent in many rules because it is frequent; lift is therefore more informative than confidence for assessing whether those rules reflect a real association.
